# Notebook 05_08: Training Best Practices

**Learning Objectives:**
- Use HuggingFace's `Trainer` API for streamlined model training
- Configure `TrainingArguments` for learning rate schedules, warmup, and weight decay
- Apply gradient accumulation to train with larger effective batch sizes on limited hardware
- Enable mixed precision (FP16/BF16) training for speed and memory savings
- Implement early stopping and checkpointing to prevent overfitting
- Compare training configurations systematically with proper evaluation

**Prerequisites:** Notebooks 01_07/01_08 (Fine-tuning concepts), 05_02 (Performance)

## Prerequisites

### Hardware Requirements

| Requirement | CPU Path | GPU Path |
|-------------|----------|----------|
| **RAM** | 8 GB minimum | 8 GB |
| **VRAM** | N/A | 4 GB+ (RTX 4080 ideal) |
| **Storage** | 1 GB free | 1 GB free |
| **Time** | 5-15 min per experiment | 1-3 min per experiment |

### Software Requirements

```bash
pip install transformers datasets evaluate accelerate
```

## Expected Behaviors

### Training Runs
- Each training experiment takes 1-5 minutes on CPU, under 1 minute on GPU
- Loss should decrease steadily over epochs
- Evaluation accuracy should improve then plateau
- Early stopping will halt training when validation loss stops improving

### Common Observations
- **Warmup helps**: Models with learning rate warmup typically converge more stably
- **Gradient accumulation**: Simulates larger batch sizes without more memory
- **Mixed precision**: ~1.5-2x speedup on GPU, negligible impact on CPU
- **Overfitting**: Without regularization, small datasets overfit quickly

### Common Issues
- **CUDA out of memory**: Reduce `per_device_train_batch_size` or enable gradient accumulation
- **Slow training on CPU**: Expected — reduce `num_train_epochs` or use a smaller model
- **Warning about unused columns**: Safe to ignore — Trainer auto-removes extra columns

## Overview

The HuggingFace `Trainer` API handles the training loop, evaluation, logging, checkpointing, and distributed training — so you can focus on *what* to train rather than *how*. This notebook covers the most impactful training best practices:

| Practice | What It Does | When to Use |
|----------|-------------|-------------|
| **Learning Rate Schedule** | Adjusts LR during training (warmup + decay) | Always |
| **Weight Decay** | L2 regularization to prevent overfitting | Always |
| **Gradient Accumulation** | Simulates larger batch sizes | Limited GPU memory |
| **Mixed Precision (FP16)** | Uses half-precision for speed | GPU training |
| **Early Stopping** | Stops when validation loss plateaus | Preventing overfitting |
| **Checkpointing** | Saves model at regular intervals | Long training runs |
| **Evaluation Strategy** | Evaluate during training, not just at the end | Always |

We will train a text classifier (DistilBERT on a sentiment dataset) using different configurations and compare the results.

## Setup and Installation

In [ ]:
import os
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset
import evaluate

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Model and Data Selection

We use DistilBERT for text classification — small enough to train quickly but large enough for meaningful experiments. The dataset is a subset of the IMDB sentiment dataset.

In [ ]:
# Option 1: Small model (CPU-friendly, ~268 MB)
MODEL_NAME = "distilbert-base-uncased"

# Option 2: Larger model (GPU recommended, ~440 MB)
# MODEL_NAME = "bert-base-uncased"

# Training configuration
NUM_LABELS = 2          # Binary sentiment
MAX_LENGTH = 128        # Token sequence length
TRAIN_SAMPLES = 1000    # Subset for fast experiments
EVAL_SAMPLES = 200      # Subset for evaluation

print(f"Model: {MODEL_NAME}")
print(f"Train samples: {TRAIN_SAMPLES}")
print(f"Eval samples: {EVAL_SAMPLES}")

## Data Preparation

We load the IMDB dataset and take a small subset for fast experimentation. In production, you would use the full dataset, but for learning the Trainer API, a small subset lets us iterate quickly.

In [ ]:
print("Loading IMDB dataset...")
dataset = load_dataset("imdb")

# Take subsets for fast training
train_dataset = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SAMPLES))
eval_dataset = dataset["test"].shuffle(seed=SEED).select(range(EVAL_SAMPLES))

print(f"Train: {len(train_dataset)} samples")
print(f"Eval:  {len(eval_dataset)} samples")
print(f"\nSample: {train_dataset[0]['text'][:100]}...")
print(f"Label:  {train_dataset[0]['label']} ({'positive' if train_dataset[0]['label'] == 1 else 'negative'})")

In [ ]:
# Tokenize the datasets
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_function(examples: dict) -> dict:
    """Tokenize a batch of text examples.

    Args:
        examples: Dictionary with 'text' key containing strings.

    Returns:
        Tokenized dictionary with input_ids, attention_mask.
    """
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )


print("Tokenizing datasets...")
train_tokenized = train_dataset.map(tokenize_function, batched=True, desc="Tokenizing train")
eval_tokenized = eval_dataset.map(tokenize_function, batched=True, desc="Tokenizing eval")

# Set format for PyTorch
train_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
eval_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"Tokenized train columns: {train_tokenized.column_names}")
print(f"Sample input shape: {train_tokenized[0]['input_ids'].shape}")

## Evaluation Metrics

We define a `compute_metrics` function that the Trainer calls during evaluation. This lets us track accuracy alongside the default loss metric.

In [ ]:
accuracy_metric = evaluate.load("accuracy")


def compute_metrics(eval_pred: tuple) -> dict[str, float]:
    """Compute accuracy from model predictions.

    Args:
        eval_pred: Tuple of (logits, labels) from Trainer.

    Returns:
        Dictionary with accuracy score.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)


print("Metrics function ready: accuracy")

## Experiment 1: Baseline Training

Our first experiment uses minimal configuration — just enough to train a model. This establishes the baseline that we will improve upon.

In [ ]:
def load_fresh_model() -> AutoModelForSequenceClassification:
    """Load a fresh copy of the model for each experiment.

    Returns:
        Freshly initialized model for sequence classification.
    """
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
    )
    return model


def run_experiment(
    experiment_name: str,
    training_args: TrainingArguments,
    callbacks: list | None = None,
) -> dict[str, float]:
    """Run a training experiment and return results.

    Args:
        experiment_name: Name for this experiment.
        training_args: HuggingFace TrainingArguments.
        callbacks: Optional list of Trainer callbacks.

    Returns:
        Dictionary with experiment name, eval_loss, eval_accuracy, and train_time.
    """
    print(f"\n{'='*60}")
    print(f"  Experiment: {experiment_name}")
    print(f"{'='*60}\n")

    model = load_fresh_model()

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=eval_tokenized,
        compute_metrics=compute_metrics,
        callbacks=callbacks or [],
    )

    start_time = time.perf_counter()
    trainer.train()
    train_time = time.perf_counter() - start_time

    eval_results = trainer.evaluate()

    print(f"\nResults for {experiment_name}:")
    print(f"  Eval Loss:     {eval_results['eval_loss']:.4f}")
    print(f"  Eval Accuracy: {eval_results['eval_accuracy']:.2%}")
    print(f"  Train Time:    {train_time:.1f}s")

    # Cleanup
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "Experiment": experiment_name,
        "Eval Loss": eval_results["eval_loss"],
        "Eval Accuracy": eval_results["eval_accuracy"],
        "Train Time (s)": round(train_time, 1),
    }


# Store all experiment results
all_results: list[dict[str, float]] = []

In [ ]:
# Experiment 1: Baseline (minimal config)
OUTPUT_DIR = "./training_output/baseline"

baseline_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    seed=SEED,
    report_to="none",        # Disable wandb/tensorboard
    disable_tqdm=False,
)

result = run_experiment("Baseline", baseline_args)
all_results.append(result)

## Experiment 2: Learning Rate Schedule with Warmup

A learning rate warmup gradually increases the LR from 0 to the target value over the first N steps. This helps prevent early training instability — when model weights are random, large gradient updates can push the model into bad regions. After warmup, a cosine or linear decay schedule gradually reduces the LR, allowing the model to settle into a better minimum.

### Common Schedules

| Schedule | Behavior | Best For |
|----------|----------|----------|
| **Linear** | Linear warmup, then linear decay to 0 | General purpose |
| **Cosine** | Linear warmup, then cosine curve to 0 | Longer training |
| **Constant** | Warmup, then constant LR | Short fine-tuning |
| **Polynomial** | Warmup, then polynomial decay | NLP fine-tuning |

In [ ]:
# Experiment 2: With LR warmup + cosine schedule
warmup_args = TrainingArguments(
    output_dir="./training_output/warmup_cosine",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=2e-5,              # Explicit LR
    lr_scheduler_type="cosine",      # Cosine decay after warmup
    warmup_ratio=0.1,                # 10% of steps as warmup
    seed=SEED,
    report_to="none",
    disable_tqdm=False,
)

result = run_experiment("LR Warmup + Cosine", warmup_args)
all_results.append(result)

## Experiment 3: Weight Decay (Regularization)

Weight decay adds an L2 penalty to model weights during optimization, preventing them from growing too large. This acts as regularization, reducing overfitting — especially important when your training set is small relative to the model size (as in our case with 1,000 examples and ~67M parameters).

The `Trainer` implements *decoupled weight decay* (AdamW), which separates the decay from the gradient update. Typical values range from 0.01 to 0.1.

In [ ]:
# Experiment 3: Weight decay + warmup + cosine
weight_decay_args = TrainingArguments(
    output_dir="./training_output/weight_decay",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,               # L2 regularization
    seed=SEED,
    report_to="none",
    disable_tqdm=False,
)

result = run_experiment("Weight Decay (0.01)", weight_decay_args)
all_results.append(result)

## Experiment 4: Gradient Accumulation

Gradient accumulation lets you simulate a larger batch size without requiring more GPU memory. Instead of updating weights after every batch, gradients are accumulated over N batches and then applied. This is essential for training on hardware with limited memory.

**Effective batch size** = `per_device_train_batch_size` x `gradient_accumulation_steps` x `num_gpus`

For example: batch size 8 with 4 accumulation steps = effective batch size 32.

In [ ]:
# Experiment 4: Gradient accumulation (effective batch 32)
grad_accum_args = TrainingArguments(
    output_dir="./training_output/grad_accum",
    num_train_epochs=3,
    per_device_train_batch_size=8,        # Smaller batch
    gradient_accumulation_steps=4,         # Accumulate 4 batches
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    seed=SEED,
    report_to="none",
    disable_tqdm=False,
)

result = run_experiment("Grad Accumulation (eff. 32)", grad_accum_args)
all_results.append(result)

## Experiment 5: Mixed Precision Training (FP16 / BF16)

Mixed precision training uses FP16 (or BF16) for most computations while keeping a master copy of weights in FP32. This provides:
- **~1.5-2x speedup** on modern GPUs (Tensor Cores)
- **~50% less memory** per batch
- **Minimal accuracy impact** (loss scaling prevents underflow)

On CPU, mixed precision has little benefit, but we enable it here for completeness. The Trainer automatically handles loss scaling.

In [ ]:
# Experiment 5: Mixed precision
# Use FP16 on CUDA, BF16 if supported, skip on CPU
use_fp16 = torch.cuda.is_available()
use_bf16 = False
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    use_bf16 = True
    use_fp16 = False

mixed_args = TrainingArguments(
    output_dir="./training_output/mixed_precision",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=use_fp16,                    # FP16 mixed precision
    bf16=use_bf16,                    # BF16 if available
    seed=SEED,
    report_to="none",
    disable_tqdm=False,
)

precision_label = "BF16" if use_bf16 else ("FP16" if use_fp16 else "FP32 (CPU)")
result = run_experiment(f"Mixed Precision ({precision_label})", mixed_args)
all_results.append(result)

## Experiment 6: Early Stopping

Early stopping monitors a metric (typically validation loss) and halts training when it stops improving. This prevents overfitting and saves compute time. The `patience` parameter controls how many evaluation steps without improvement are tolerated before stopping.

The Trainer integrates early stopping through the `EarlyStoppingCallback`. You must set `load_best_model_at_end=True` so the final model is the best checkpoint, not the last one.

In [ ]:
# Experiment 6: Early stopping (more epochs, will stop early if overfitting)
early_stop_args = TrainingArguments(
    output_dir="./training_output/early_stopping",
    num_train_epochs=10,                  # More epochs — early stopping will cut short
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    load_best_model_at_end=True,          # Required for early stopping
    metric_for_best_model="eval_loss",    # Monitor this metric
    greater_is_better=False,              # Lower loss is better
    seed=SEED,
    report_to="none",
    disable_tqdm=False,
)

early_stop_callback = EarlyStoppingCallback(
    early_stopping_patience=2,    # Stop after 2 epochs without improvement
)

result = run_experiment("Early Stopping (patience=2)", early_stop_args, callbacks=[early_stop_callback])
all_results.append(result)

## Results Comparison

Let's bring all experiments together in a single table. This shows how each training practice impacts model quality and training efficiency.

In [ ]:
print("=== EXPERIMENT COMPARISON ===\n")
results_df = pd.DataFrame(all_results)
results_df["Eval Accuracy"] = results_df["Eval Accuracy"].apply(lambda x: f"{x:.2%}")
results_df["Eval Loss"] = results_df["Eval Loss"].apply(lambda x: f"{x:.4f}")
print(results_df.to_string(index=False))

print("\n--- Observations ---")
print("- LR warmup + cosine schedule typically stabilizes training")
print("- Weight decay prevents overfitting on small datasets")
print("- Gradient accumulation achieves similar quality with less memory")
print("- Mixed precision speeds up GPU training with minimal quality impact")
print("- Early stopping saves compute by halting when validation plateaus")

## Recommended Configuration

Based on the experiments above, here is a recommended `TrainingArguments` configuration that combines all best practices. You can use this as a starting template and adjust based on your specific task and hardware.

In [ ]:
# Recommended starting configuration for fine-tuning
recommended_config = {
    "Learning Rate": "2e-5 (BERT-family) or 5e-5 (smaller models)",
    "LR Scheduler": "cosine (general) or linear (short runs)",
    "Warmup Ratio": "0.06 - 0.1 (6-10% of total steps)",
    "Weight Decay": "0.01 - 0.1",
    "Batch Size": "16-32 (or use gradient accumulation)",
    "Gradient Accumulation": "Increase until effective batch ~32",
    "Mixed Precision": "FP16 on GPU (always), skip on CPU",
    "Epochs": "3-5 for fine-tuning, more with early stopping",
    "Early Stopping Patience": "2-3 epochs",
    "Evaluation Strategy": "per epoch (or every N steps for large datasets)",
    "Save Strategy": "match eval strategy + load_best_model_at_end=True",
}

print("=== RECOMMENDED TRAINING CONFIGURATION ===\n")
for setting, value in recommended_config.items():
    print(f"  {setting:30s} : {value}")

print("\n--- Complete TrainingArguments Example ---\n")
print('''TrainingArguments(
    output_dir="./output",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,     # effective batch 32
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,                         # GPU only
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=1103,
)''')

## Cleanup

Let's remove the training output directories created during our experiments.

In [ ]:
import shutil

output_base = "./training_output"
if os.path.exists(output_base):
    shutil.rmtree(output_base)
    print(f"Cleaned up {output_base}/ directory.")
else:
    print("No training output to clean up.")

## Exercises

1. **Learning Rate Sweep**: Try learning rates of 1e-5, 2e-5, 5e-5, and 1e-4. Plot eval accuracy vs learning rate to find the optimal value.

2. **Scheduler Comparison**: Compare `linear`, `cosine`, `cosine_with_restarts`, and `polynomial` schedulers. Which gives the best final accuracy?

3. **Full Dataset Training**: Increase `TRAIN_SAMPLES` to 5000 or use the full IMDB train set. Does early stopping trigger later? Do the relative rankings of configurations change?

4. **Different Model**: Swap DistilBERT for `bert-base-uncased` or `roberta-base`. Do the same best practices apply?

5. **Custom Callback**: Write a custom `TrainerCallback` that logs the learning rate at each step and plots the LR schedule after training.

## Key Takeaways

- **Learning rate warmup + cosine decay** stabilizes training and often improves final accuracy
- **Weight decay (0.01)** provides essential regularization for fine-tuning on small datasets
- **Gradient accumulation** lets you simulate larger batch sizes without extra GPU memory
- **Mixed precision (FP16/BF16)** provides ~1.5-2x speedup on GPU with negligible accuracy impact
- **Early stopping + load_best_model_at_end** prevents overfitting and saves compute time

## Next Steps

- Try **Notebook 06_01**: MCP Basics (Agentic Workflows with tool-using agents)
- Explore [Trainer Documentation](https://huggingface.co/docs/transformers/main_classes/trainer)
- Read about [PEFT/LoRA](https://huggingface.co/docs/peft/) for parameter-efficient fine-tuning

## Resources

- [HuggingFace Trainer](https://huggingface.co/docs/transformers/main_classes/trainer)
- [TrainingArguments Reference](https://huggingface.co/docs/transformers/main_classes/trainer#trainingarguments)
- [Learning Rate Schedules](https://huggingface.co/docs/transformers/main_classes/optimizer_schedules)
- [Mixed Precision Training](https://huggingface.co/docs/transformers/perf_train_gpu_one#mixed-precision-training)
- [Early Stopping Guide](https://huggingface.co/docs/transformers/main_classes/callback#earlystoppingcallback)